### MC vs Data Tuning

Throughgoing Muons, Michels, and AmBe cumulative PMT charge / hit MC vs data agreement studies.

1. **Throughgoing muons** (high-energy ~GeV scale, MRD-coincident, similar track topology between events)
2. **Michel electrons** (intermediate ~50 MeV scale, decay-in-flight from dirt muons, highly anisotropic vertex distribution to stress test PMT optical model and simulation modifications)
3. **AmBe neutron candidates** (low-energy, isotropic and homogenous tuning)

Methodology follows Steven Doran's dissertation (Chapter 3 + chi² appendix). For AmBe, which has a known source position per event, MC and data are balanced by random sampling **per source position** before computing chi². For Michels and throughgoing muons, MC and data are balanced by simple random subsampling. The AmBe machinery here utilizes the final NC tuning (PMT tilting, waveform simulation, angular collection efficiencies, dead-PMT mask applied, and tuned against the total charge distribution). The Michels are used to validate the detector response (total charge and total hits distributions).

Feel free to breakup this script as you see fit - I have included all three datasets for consistency but difference cells can be omitted to look at a specific energy regime / dataset. 

Disclaimer: This script is a condensed, less-cluttered version of the MC tuning machinery used, cleaned by Claude AI. The authoritative, non-AI version actually deployed to do the tuning analysis can be found in the same repo for the AmBe dataset (`AmBe_tuning.ipynb`) and the other datasets as part of the CC baseline tuning campaign (`lib/` folder). Also keep in mind the data files have ALREADY been filtered from the BeamCluster ntuples, with data selection cuts applied. See Steven Doran's dissertation for details on the selection cuts used for all three datasets.

## 0. Setup and helper functions

In [1]:
print('Loading packages...\n')
%matplotlib osx
import sys, os
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import chi2_contingency
import scipy.stats
import pandas
from collections import defaultdict
import random

# --- plot style ---
font = {'family': 'serif', 'size': 14}
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['legend.fontsize'] = 16
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# --- PMT geometry + dead mask (used by data + MC loaders) ---
df = pandas.read_csv('lib/FullTankPMTGeometry.csv')
channel_number = []; PMT_type = []
for i in range(len(df['channel_num'])):
    channel_number.append(df['channel_num'][i])
    PMT_type.append(df['PMT_type'][i])

# Updated PMT dead mask to reflect data (in case PMTs weren't filtered in ToolAnalysis)
dead_mask = {416, 343, 358, 359, 408, 366}


# ----------------------------------------------------------------------
# Helper functions used across all three samples
# ----------------------------------------------------------------------
# See Steven Doran dissertation (chi2 appendix) for more details:
# rather than scaling MC counts to data, we randomly sample. For AmBe we
# sample per source position; for Michel/Thru we sample globally.

def get_common_positions(d1, d2):
    common_keys = set(d1.keys()) & set(d2.keys())
    filtered = []
    for k in common_keys:
        if len(d1[k]) > 0 and len(d2[k]) > 0:
            filtered.append(k)
    return filtered


def balance_dataset_with_keys(pos_dict, keys, n_per_pos):
    all_entries = []
    for k in sorted(keys):
        if len(pos_dict[k]) >= n_per_pos:
            all_entries.extend(random.sample(pos_dict[k], n_per_pos))
    return all_entries


def generate_merged_bins(bins, counts_data, counts_mc, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.
    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, 'Bin edge mismatch'

    new_bins = [bins[0]]
    i = 0
    while i < len(counts_data):
        sum_d = counts_data[i]
        sum_mc = counts_mc[i]
        j = i + 1
        while (sum_d < min_counts or sum_mc < min_counts) and j < len(counts_data):
            sum_d += counts_data[j]
            sum_mc += counts_mc[j]
            j += 1
        new_bins.append(bins[j])
        i = j
    return new_bins


def balance_samples(list1, list2):
    """Randomly subsample both lists to equal length."""
    list1 = list(list1)
    list2 = list(list2)
    n = min(len(list1), len(list2))
    return random.sample(list1, n), random.sample(list2, n)


print('done')

Loading packages...

done


## 1. Load data

### 1a. Michel data

In [2]:
# extracted Michel data file --> adjust if needed
filename = 'lib/Michels_v1.root'

rootdata = uproot.open(filename)
TD = rootdata['Michels']
CPE_data = TD['cluster_pe'].array()
CCB_data = TD['cluster_Qb'].array()
CT_data  = TD['cluster_time'].array()
CH_data  = TD['cluster_hits'].array()
T_data   = TD['hitT'].array()
PE_data  = TD['hitPE'].array()
ID_data  = TD['hitPMTID'].array()

print('Michel data loaded')

Michel data loaded


### 1b. AmBe data

Output from `extract_neutron_candidates.py` (https://github.com/S81D/AmBe_neutron_candidates/blob/main/extract_neutron_candidates.py). Based on AmBe v2.0 (summer–fall 2023). Port 3, Y = +100 cm is omitted because AmBe v2.0 did not take proper data at that location. Also Port 4 is flipped in the code to represent the true position within the tank. The dead-PMT mask is also applied here.

In [3]:
# adjust if needed
data_file = 'lib/neutron_candidates_v5.root'

# ----------------------------------------------------- #

cluster_time = []; cluster_charge = []; cluster_QB = []; cluster_hits = []
hit_times = []; hit_charges = []; hit_ids = []
source_position = [[], [], []]      # x, y, z

# last index in each list are for the counts
poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
        [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],  # Port 1
        [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],       # Port 4
        [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]  # Port 3


print('\nLoading AmBe event data...')

with uproot.open(data_file) as file:

    Event = file['Neutrons']
    CT  = Event['cluster_time'].array()
    CPE = Event['cluster_charge'].array()
    CCB = Event['cluster_Qb'].array()
    CH  = Event['cluster_hits'].array()
    hT  = Event['hitT'].array()
    hPE = Event['hitPE'].array()
    hID = Event['hitID'].array()
    xp  = Event['X_pos'].array()
    yp  = Event['Y_pos'].array()
    zp  = Event['Z_pos'].array()

    for i in trange(len(CT)):

        x1 = round(xp[i]); y1 = round(yp[i]); z1 = round(zp[i])

        # [0,100,102,0] (remove Port 3 +100cm) -- no good data at this location
        if y1 == 100 and x1 == 0 and z1 == 102:
            continue

        # ENABLE for Type-only study
        '''
        cc = 0; ch = 0
        for j in range(len(hPE[i])):
            indy = hID[i][j] - 332
            if PMT_type[indy] == 'Watchboy':
                cc += hPE[i][j]
                ch += 1
        if ch > 0:  # enable to remove 0 bin
            cluster_charge.append(cc)
            cluster_hits.append(ch)
        #'''

        # apply the dead mask
        cpe_sum = sum(hPE[i][j] for j in range(len(hID[i])) if hID[i][j] not in dead_mask)
        ch_sum  = sum(1          for j in range(len(hID[i])) if hID[i][j] not in dead_mask)
        cluster_charge.append(cpe_sum)
        cluster_hits.append(ch_sum)

        # indent if Type-only block is uncommented ^
        cluster_time.append(CT[i])
        #cluster_charge.append(CPE[i])   # comment if using Type-only
        cluster_QB.append(CCB[i])
        #cluster_hits.append(CH[i])      # comment if using Type-only
        hit_times.append(hT[i])
        hit_charges.append(hPE[i])
        hit_ids.append(hID[i])

        # In AmBe v2.0 (2023), Port 4 is backwards in X (really +75 cm)
        if x1 == -75:
            source_position[0].append(75)
        else:
            source_position[0].append(x1)
        source_position[1].append(y1)
        source_position[2].append(z1)


print('----------------------------------------------------------------\n')
print('We have: ', len(cluster_time), ' total AmBe neutron candidates\n')
print('done')


Loading AmBe event data...


100%|████████████████████████████████████| 67940/67940 [02:42<00:00, 418.38it/s]

----------------------------------------------------------------

We have:  65992  total AmBe neutron candidates

done


### 1c. Throughgoing muon data

It is important to note that while the AmBe + Michel data are well tuned for the NC tuning analysis, the throughgoing muons display a ~10% discrepency in the total charge between data and MC. This is due to the uncertainties in the PMT tilt angle compounding with higher charge. See Steven Doran's dissertation for more details.

In [5]:
# adjust if needed
filename_thru = 'lib/throughgoing_muons_v1.root'

rootdata_thru = uproot.open(filename_thru)
TD_thru = rootdata_thru['Muons']
CPE_thru_data       = TD_thru['cluster_pe'].array()
CCB_thru_data       = TD_thru['cluster_Qb'].array()
CT_thru_data        = TD_thru['cluster_time'].array()
CH_thru_data        = TD_thru['cluster_hits'].array()
Down_thru_data      = TD_thru['downstream_pe'].array()
Up_thru_data        = TD_thru['upsteam_pe'].array()
T_thru_data         = TD_thru['hitT'].array()
PE_thru_data        = TD_thru['hitPE'].array()
ID_thru_data        = TD_thru['hitPMTID'].array()
ID_number_thru_data = TD_thru['hitID'].array()

print('Thru-muon data loaded')

Thru-muon data loaded


## 2. Load MC samples

### 2a. Michel MC

Dirt muon selection followed by Michel-candidate selection in a single passthrough the simulated cluster tree.

Expect this cell to take some time to run (~5 minutes).

In [6]:
# directories to be used (adjust if needed)
MC_directories = [
    'simulated_data/QE_1.50_HM_WM_1.25/'#,
]#'simulated_data/QE_1.50_HM_1.50_2/'          # file for extra statistics

# assemble full paths
file_paths = []
for MC_dir in MC_directories:
    full_paths = [os.path.join(MC_dir, fname) for fname in os.listdir(MC_dir)
                  if os.path.isfile(os.path.join(MC_dir, fname))]
    file_paths.extend(full_paths)


charge_balance = []; total_charge = []
muon_charge = []; muon_charge_balance = []; total_hits = []; total_time = []; avg_hit = []
dirt_muon_count = 0

count_cuts = [0, 0, 0, 0, 0, 0, 0, 0, 0]


# Process each file
for count, filepath in enumerate(file_paths):
    if count % 50 == 0:
        print(round(100 * count / len(file_paths), 2), '%')

    root = uproot.open(filepath)
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CT  = T['clusterTime'].array()
    CN  = T['clusterNumber'].array()
    CH  = T['clusterHits'].array()
    X1  = T['hitX'].array()
    Y1  = T['hitY'].array()
    Z1  = T['hitZ'].array()
    T1  = T['hitT'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    TMC = T['TankMRDCoinc'].array()
    NV1 = T['NoVeto'].array()

    # need the TriggerTree for MRD activity
    Trig = root['phaseIITriggerTree']
    EN2  = Trig['eventNumber'].array()
    MRD2 = Trig['MRDhitT'].array()

    # build eventNumber -> MRDhitT map
    event_to_MRD = dict(zip(EN2, MRD2))

    # for the PMTWaveformSim tool, we skip blank events (no MCHits) but we
    # still have a mismatch between events with < 5 clustered hits and clusters

    count_cuts[0] += len(EN2)

    # loop over clusters
    for i in range(len(CEN)):

        evt = CEN[i]

        # ---- select dirt muon candidates ----

        # 1. No MRD activity
        mrd_hits = event_to_MRD.get(evt, [])
        if len(mrd_hits) != 0:
            continue

        # 2. Require Veto activity
        if NV1[i] == 1:
            continue

        # 3. No TankMRDCoinc (should already be covered)
        if TMC[i] == 1:
            print('Seems like one got through to the MRD somehow!!!')
            continue

        count_cuts[1] += 1

        # 4. 1000 < cluster PE < 4000
        if CPE[i] > 4000 or CPE[i] < 1000:
            continue

        count_cuts[2] += 1

        # 5. Prompt cluster, and can't be the only cluster (we need Michel candidates)
        if CN[i] != 0:
            if (i+1) == len(CN) or CN[i+1] == 0:  # no more clusters --> no Michel possible
                continue

        # 6. (just for sim) cluster time under 150 ns so it's a prompt event
        if CT[i] > 150:
            continue

        # 7. It's the brightest event
        brightest = CPE[i]; flag = False
        for j in range(1, len(CEN) - i):
            if CN[j] == 0:
                break
            if CPE[j] > brightest:
                flag = True
                break
        if flag == True:
            continue

        count_cuts[3] += 1

        # 8. charge barycenter downstream
        a = 0
        for j in range(len(Z1[i])):
            a += Z1[i][j] * PE1[i][j]
        if a < 0:
            continue

        count_cuts[4] += 1

        # 9. Must have 100 PMT hits
        if CH[i] < 100:
            continue

        count_cuts[5] += 1

        # CCB cuts
        if CCB[i] > 0.2 or CCB[i] < 0:
            continue

        count_cuts[6] += 1

        dirt_muon_count += 1
        muon_charge.append(CPE[i])
        muon_charge_balance.append(CCB[i])


        # ---- Michel candidates from this dirt muon ----
        for j in range(i+1, len(CEN)):
            if CN[j] == 0:   # new event
                break
            time_diff = CT[j] - CT[i]
            if 200 < time_diff < 5000:
                count_cuts[7] += 1
                if CCB[j] < 0.18:  # The charge balance cut is restricted to 0.18 for my analysis
                    count_cuts[8] += 1
                    '''    # uncomment for specific PMT-type analysis
                    a = 0; b = 0
                    for k in range(len(ID1[j])):
                        if PMT_type[ID1[j][k] - 332] == 'LUX' or PMT_type[ID1[j][k] - 332] == 'ETEL':
                            a += PE1[j][k]; b += 1
                    total_charge.append(a)
                    total_hits.append(b)
                    #'''

                    avg_hit.append(np.average(PE1[j]))
                    charge_balance.append(CCB[j])
                    total_charge.append(CPE[j])   # comment for individual PMT type
                    total_hits.append(CH[j])      # and here
                    total_time.append(CT[j])


print('\ndirt muon count: ', dirt_muon_count, 'Michel count: ', len(total_charge))
print('\ndone')

0.0 %
12.8 %
25.61 %
38.41 %
51.22 %
64.02 %
76.82 %
89.63 %

 11180 1601

done


### 2b. AmBe MC

Selection cuts mirror the AmBe data cuts, with the cluster-time window offset for the simulation latency (~1101 ns). Port 3 Y = +100 cm is skipped to mirror the data sample. Port 4's position is correctly updated.

This cell takes ~10 min to complete (for 20 port / depth positions).

In [7]:
# adjust if needed
MC_directory_approx_tilt = '../AmBe/AmBe_MC/pmt_tilting_v1/corrected_waveforms/QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25_corrected_waveforms/'
file_names_approx_tilt = [file for file in os.listdir(MC_directory_approx_tilt)
                          if os.path.isfile(os.path.join(MC_directory_approx_tilt, file))]
QE_tag_approx_tilt = '_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25'


cluster_charge_MC = []; source_position_MC = [[], [], []]; cluster_hits_MC = []

print('This cell will process 20 files in total (for each port / depth position)\n')

for file in file_names_approx_tilt:

    root = uproot.open(MC_directory_approx_tilt + file)
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CH  = T['clusterHits'].array()
    CT  = T['clusterTime'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    HT1 = T['hitT'].array()
    CN  = T['clusterNumber'].array()

    # Port 5
    if   file == 'MC_AmBe_port5_z100'      + QE_tag_approx_tilt + '.root': xp = 0;  yp =  100; zp = 0
    elif file == 'MC_AmBe_port5_z50'       + QE_tag_approx_tilt + '.root': xp = 0;  yp =   50; zp = 0
    elif file == 'MC_AmBe_port5_z0'        + QE_tag_approx_tilt + '.root': xp = 0;  yp =    0; zp = 0
    elif file == 'MC_AmBe_port5_zminus50'  + QE_tag_approx_tilt + '.root': xp = 0;  yp =  -50; zp = 0
    elif file == 'MC_AmBe_port5_zminus100' + QE_tag_approx_tilt + '.root': xp = 0;  yp = -100; zp = 0
    # Port 1
    elif file == 'MC_AmBe_port1_z100'      + QE_tag_approx_tilt + '.root': xp = 0;  yp =  100; zp = -75
    elif file == 'MC_AmBe_port1_z50'       + QE_tag_approx_tilt + '.root': xp = 0;  yp =   50; zp = -75
    elif file == 'MC_AmBe_port1_z0'        + QE_tag_approx_tilt + '.root': xp = 0;  yp =    0; zp = -75
    elif file == 'MC_AmBe_port1_zminus50'  + QE_tag_approx_tilt + '.root': xp = 0;  yp =  -50; zp = -75
    elif file == 'MC_AmBe_port1_zminus100' + QE_tag_approx_tilt + '.root': xp = 0;  yp = -100; zp = -75
    # Port 4 (x position correctly implemented)
    elif file == 'MC_AmBe_port4_z100'      + QE_tag_approx_tilt + '.root': xp = 75; yp =  100; zp = 0
    elif file == 'MC_AmBe_port4_z50'       + QE_tag_approx_tilt + '.root': xp = 75; yp =   50; zp = 0
    elif file == 'MC_AmBe_port4_z0'        + QE_tag_approx_tilt + '.root': xp = 75; yp =    0; zp = 0
    elif file == 'MC_AmBe_port4_zminus50'  + QE_tag_approx_tilt + '.root': xp = 75; yp =  -50; zp = 0
    elif file == 'MC_AmBe_port4_zminus100' + QE_tag_approx_tilt + '.root': xp = 75; yp = -100; zp = 0
    # Port 3 (skip +100 cm to reflect data)
    elif file == 'MC_AmBe_port3_z100'      + QE_tag_approx_tilt + '.root':
        continue
    elif file == 'MC_AmBe_port3_z50'       + QE_tag_approx_tilt + '.root': xp = 0;  yp =   50; zp = 102
    elif file == 'MC_AmBe_port3_z0'        + QE_tag_approx_tilt + '.root': xp = 0;  yp =    0; zp = 102
    elif file == 'MC_AmBe_port3_zminus50'  + QE_tag_approx_tilt + '.root': xp = 0;  yp =  -50; zp = 102
    elif file == 'MC_AmBe_port3_zminus100' + QE_tag_approx_tilt + '.root': xp = 0;  yp = -100; zp = 102
    else:
        print('INCORRECT FILE NAME!!!')


    # APPLY SELECTION CUTS (same as data; timing window shifted by simulation latency)
    for i in trange(len(CPE), desc=file):

        if CPE[i] < 70 and (67000 - 1101) > CT[i] > (2000 - 1101) and CCB[i] < 0.4:

            # only cluster in the event (can adjust for multiplicity / background study)
            if CN[i] != 0:
                continue
            else:    # zeroth cluster, make sure it's the only cluster
                if i + 1 < len(CN):
                    if CN[i + 1] != 0:
                        continue

            ''' comment for PMT-type analysis
            cc = 0
            for j in range(len(PE1[i])):
                indy = ID1[i][j] - 332
                if PMT_type[indy] == 'Watchboy':
                    cc += PE1[i][j]
            if cc > 0:       # for zero bin
            #'''

            # apply dead mask
            cpe_sum = sum(PE1[i][j] for j in range(len(ID1[i])) if ID1[i][j] not in dead_mask)
            ch_sum  = sum(1          for j in range(len(ID1[i])) if ID1[i][j] not in dead_mask)
            cluster_charge_MC.append(cpe_sum)
            cluster_hits_MC.append(ch_sum)

            # requires indentation if doing PMT-type analysis
            #cluster_charge_MC.append(CPE[i])   # comment for type-only
            #cluster_charge_MC[geo].append(cc)  # uncomment for type-only
            #cluster_hits_MC.append(CH[i])
            source_position_MC[0].append(xp)
            source_position_MC[1].append(yp)
            source_position_MC[2].append(zp)


print('\ndone')

MC_AmBe_port3_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15365/15
MC_AmBe_port1_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17164/17
MC_AmBe_port1_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17280/17280 [0
MC_AmBe_port3_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15515/15515 [0
MC_AmBe_port5_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17465/17
MC_AmBe_port3_zminus100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 13539/1
MC_AmBe_port4_z50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17422/17422 [
MC_AmBe_port1_zminus100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15916/1
MC_AmBe_port1_z50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17458/17458 [
MC_AmBe_port5_z100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17670/17670 
MC_AmBe_port4_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17326/17326 [0
MC_AmBe_port4_z100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17147/17147 
MC_AmBe_port3_z50_QE_1.50_WB


done


### 2c. Throughgoing muon MC

In [21]:
# adjust paths as needed, expect this script to take ~5 min

# for a single file
#file = 'simulated_data/MC_dirt_muons_QE_1.25_0_500k.ntuple.root'

# for a group of files
MC_directory = 'simulated_data/thru/QE_1.5_HM_1.25/'
file_names = [file for file in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, file))]

charge_balance_thru = []; total_charge_thru = []; total_hits_thru = []
downstream_charge = []; upstream_charge = []; avg_hit_thru = []
thru_muon_count = 0

Top_charge = []; Bottom_charge = []

count = 0
for file in file_names:         # disable for a single file

    if count % 5 == 0:
        print(round(100*(count/len(file_names)),2), '%')
    count += 1

    # indent for collection of files
    root = uproot.open(MC_directory + file)       # change to just 'file' for a single file
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CT  = T['clusterTime'].array()
    CN  = T['clusterNumber'].array()
    CH  = T['clusterHits'].array()
    X1  = T['hitX'].array()
    Y1  = T['hitY'].array()
    Z1  = T['hitZ'].array()
    T1  = T['hitT'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    TMC = T['TankMRDCoinc'].array()
    NV1 = T['NoVeto'].array()

    # need the TriggerTree for MRD activity
    Trig = root['phaseIITriggerTree']
    EN2      = Trig['eventNumber'].array()
    MRD2     = Trig['numMRDTracks'].array()
    MRD_thru = Trig['MRDThrough'].array()

    # build eventNumber -> MRD map
    event_to_MRD      = dict(zip(EN2, MRD2))
    event_to_MRD_thru = dict(zip(EN2, MRD_thru))

    # loop over clusters
    for i in range(len(CEN)):

        evt = CEN[i]

        # ---- select through-going muon candidates ----

        # 1. MRD activity (coincidence, track, and through-going)
        mrd_track = event_to_MRD.get(evt, [])       # 0 if missing
        mrd_thru  = event_to_MRD_thru.get(evt, [])  # [] / [True] / [False]

        if TMC[i] == 0:           # must have Tank/MRD coincidence
            continue
        if mrd_track != 1:        # not a single track
            continue
        if len(mrd_thru) == 0:    # not a through-going track
            continue
        if mrd_thru[0] != True:
            continue

        # 2. Require Veto activity
        if NV1[i] == 1:
            continue

        # 3. Prompt cluster (within first 150 ns)
        if CN[i] != 0:
            continue
        if CT[i] > 150:
            continue

        # 4. Brightest cluster in event
        brightest = CPE[i]; flag = False
        for j in range(1, len(CEN) - i):
            if CN[j] == 0:
                break
            if CPE[j] > brightest:
                flag = True
                break
        if flag == True:
            continue

        # 5. Charge balance < 0.2
        if CCB[i] > 0.2 or CCB[i] < 0:
            continue

        # 6. Must have 100 PMT hits
        if CH[i] < 100:
            continue

        # 7. Charge barycenter downstream
        a = 0; down_c = 0; up_c = 0
        for j in range(len(Z1[i])):
            a += Z1[i][j]*PE1[i][j]
            if Z1[i][j] > 0:
                down_c += PE1[i][j]
            else:
                up_c += PE1[i][j]
        if a < 0:
            continue

        '''  uncomment for specific PMT-type analysis
        e = 0; f = 0
        gg = 0; zz = 0
        for k in range(len(ID1[i])):
            if PMT_type[ID1[i][k] - 332] == 'Hamamatsu':
                e += PE1[i][k]; f += 1
                if Y1[i][k] > 0 and Z1[i][k] > 0:
                    gg += PE1[i][k]   # Top and downstream
                elif Y1[i][k] < 0 and Z1[i][k] > 0:
                    zz += PE1[i][k]   # Bottom and downstream

        if f > 0:   # indent below if uncommenting
            total_charge_thru.append(e)
            total_hits_thru.append(f)
            Top_charge.append(gg); Bottom_charge.append(zz)
        #'''

        thru_muon_count += 1

        avg_hit_thru.append(np.average(PE1[i]))
        charge_balance_thru.append(CCB[i])
        total_charge_thru.append(CPE[i])   # comment for PMT-type
        total_hits_thru.append(CH[i])      # comment here too

        downstream_charge.append(down_c)
        upstream_charge.append(up_c)


print('\n', thru_muon_count, 'throughgoing muon candidates')
print('\ndone')

0.0 %
10.0 %
20.0 %
30.0 %
40.0 %
50.0 %
60.0 %
70.0 %
80.0 %
90.0 %

 10289 throughgoing muon candidates

done


## 3. Cumulative MC vs Data agreement

Each sample produces a histogram comparison + ratio subplot + chi²/ndf. Bins are merged adaptively to ensure ≥10 counts per bin in both MC and data.

### 3a. AmBe

Cumulative cluster charge per-source-position random sampling. The 20-position AmBe sample reduces the influence of any single source location on the fit, providing an isotropic and homegenous detector response test.

In [12]:
# this snippet will produce a cumulative PMT charge plot and provide chi-sq agreements

plot_save_dir = 'plots/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False    # enable for verbosity

# --- AmBe plot details ---
geo_colors = 'dodgerblue'
bin_size_AmBe = 1.0                              # default: 1.0 (0.5 for Watchboys / HM)
binning_AmBe = np.arange(5, 70, bin_size_AmBe)   # default: 5, 70 (0:30 for WB / HM)
xlim_low_AmBe = 0; xlim_high_AmBe = 75           # plot xlim window: default [0,75] (0:35 for WB / HM)

# no Port 3 +100cm data was loaded in, so it should not be included.
# "mc_keys_approx_tilt" should have only 19 entries.

# # # # # # # # # # # # # # # # # # # # # # # # # #

data_dict = defaultdict(list)
mc_dict_approx_tilt = defaultdict(list)

# Prepare full sample
data_all = []; mc_all_approx_tilt = []

# we must ensure equal counts at each position source for all MC and data
# Key: (x, y, z)
for i in range(len(cluster_charge)):
    key = (source_position[0][i], source_position[1][i], source_position[2][i])
    data_dict[key].append(cluster_charge[i])

for i in range(len(cluster_charge_MC)):
    key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
    mc_dict_approx_tilt[key].append(cluster_charge_MC[i])

mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
data_keys = list(data_dict.keys())

# Find shared, valid keys
valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

# Get minimum across all datasets for just those positions
min_data = min(len(data_dict[k]) for k in valid_keys)
min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
min_common = min(min_data, min_mc_approx_tilt)

# Now sample equally per position from these valid keys
data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

# Sanity check
assert len(data_all) == len(mc_all_approx_tilt), 'Data and MC not balanced after enforced key matching!'

if verbose:
    print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


# MC counts and bins
counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)
counts_mc_approx_tilt = raw_counts_mc_approx_tilt

fill_bins = generate_merged_bins(binning_AmBe, counts_data, counts_mc_approx_tilt, min_counts=10)

# re-histogram with updated binning
counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)
counts_mc_approx_tilt = raw_counts_mc_approx_tilt

if verbose:
    print('\nMerged bin edges:')
    for i in range(len(fill_bins) - 1):
        print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


# Errors
xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
data_errors = np.sqrt(counts_data)
mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


# Compute chi-squared (combined MC + Data errors)
chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                          (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
ndf = len(counts_data) - 1
if verbose:
    print(f'\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}')


# residuals + errors
mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; x_info = []

for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue
    x_info.append(binscenters[i])

    if counts_mc_approx_tilt[i] == 0:
        mc_approx_tilt_data.append(0)
        mc_errors_approx_tilt_plot.append(0)
    else:
        ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
        error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
        )
        mc_approx_tilt_data.append(ratio_mc_approx_tilt)
        mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)


# ---------- plotting ----------
fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data AmBe neutrons', fontsize=16)
ax2.set_xlabel('Cluster charge [p.e.]', fontsize=14)
ax1.set_ylabel(f'clusters / {bin_size_AmBe} p.e. bin', fontsize=14)
ax2.set_ylabel('MC / Data', fontsize=14)

ax1.set_xlim([xlim_low_AmBe, xlim_high_AmBe])
ax2.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_ylim([0, 2])

# MC histogram
ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt,
         histtype='step', color=geo_colors, label='MC', linewidth=2)
ax1.fill_between(
    binscenters,
    counts_mc_approx_tilt - mc_errors_approx_tilt,
    counts_mc_approx_tilt + mc_errors_approx_tilt,
    step='mid', color=geo_colors, alpha=0.3
)

# data histogram
ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

# residual subplot
ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot,
             markersize=1, fmt='s', color=geo_colors, label='MC')
ax2.axhline(1, linestyle='dashed', color='k')

ax1.text(0.67, 0.52, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=10)
ax1.text(0.67, 0.42, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}",
         transform=ax1.transAxes, fontsize=10, color='k')
ax1.legend(fontsize=12, frameon=False, loc='upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_AmBe_neutrons_cluster_pe_' + plot_save_name_base + '.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('\ndone')


done


### 3b. AmBe — cumulative PMT hits

In [13]:
# this snippet will produce a cumulative PMT hits plot
# the hits are slightly mis-aligned (~5%), which is absorbed as a systematic in the NCQE analysis

plot_save_dir = 'plots/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False

# --- AmBe ---
geo_colors = 'dodgerblue'
bin_size_AmBe = 2.0
binning_AmBe = np.arange(6, 36, bin_size_AmBe)   # default: 5, 70 (0:30 for WB / HM)
xlim_low_AmBe = 6; xlim_high_AmBe = 40

# # # # # # # # # # # # # # # # # # # # # # # # # #
data_dict = defaultdict(list)
mc_dict_approx_tilt = defaultdict(list)

data_all = []; mc_all_approx_tilt = []

# Key: (x, y, z)
for i in range(len(cluster_charge)):
    key = (source_position[0][i], source_position[1][i], source_position[2][i])
    data_dict[key].append(cluster_hits[i])

for i in range(len(cluster_charge_MC)):
    key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
    mc_dict_approx_tilt[key].append(cluster_hits_MC[i])

mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
data_keys = list(data_dict.keys())

valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

min_data = min(len(data_dict[k]) for k in valid_keys)
min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
min_common = min(min_data, min_mc_approx_tilt)

data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

assert len(data_all) == len(mc_all_approx_tilt), 'Data and MC not balanced after enforced key matching!'

if verbose:
    print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)
counts_mc_approx_tilt = raw_counts_mc_approx_tilt

fill_bins = generate_merged_bins(binning_AmBe, counts_data, counts_mc_approx_tilt, min_counts=10)

counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)
counts_mc_approx_tilt = raw_counts_mc_approx_tilt

if verbose:
    print('\nMerged bin edges:')
    for i in range(len(fill_bins) - 1):
        print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
data_errors = np.sqrt(counts_data)
mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                          (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
ndf = len(counts_data) - 1
if verbose:
    print(f'\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}')


mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; x_info = []

for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue
    x_info.append(binscenters[i])

    if counts_mc_approx_tilt[i] == 0:
        mc_approx_tilt_data.append(0)
        mc_errors_approx_tilt_plot.append(0)
    else:
        ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
        error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
        )
        mc_approx_tilt_data.append(ratio_mc_approx_tilt)
        mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)


# ---------- plotting ----------
fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data AmBe neutrons', fontsize=16)
ax2.set_xlabel('PMT Hits', fontsize=14)
ax1.set_ylabel('clusters / PMT hit bin', fontsize=14)
ax2.set_ylabel('MC / Data', fontsize=14)

ax1.set_xlim([xlim_low_AmBe, xlim_high_AmBe])
ax2.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_ylim([0, 2])

ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt,
         histtype='step', color=geo_colors, label='MC', linewidth=2)
ax1.fill_between(
    binscenters,
    counts_mc_approx_tilt - mc_errors_approx_tilt,
    counts_mc_approx_tilt + mc_errors_approx_tilt,
    step='mid', color=geo_colors, alpha=0.3
)

ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot,
             markersize=1, fmt='s', color=geo_colors, label='MC')
ax2.axhline(1, linestyle='dashed', color='k')

ax1.text(0.67, 0.52, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=10)
ax1.text(0.67, 0.42, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}",
         transform=ax1.transAxes, fontsize=10, color='k')
ax1.legend(fontsize=12, frameon=False, loc='upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_AmBe_neutrons_cluster_hits_' + plot_save_name_base + '.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('\ndone')


done


### 3c. Michel — cumulative cluster charge

In [18]:
# Michel plot

plot_save_dir = 'plots/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False

plot_color_Michel = 'tab:orange'
bin_size_Michel = 25     # default 20
binning_Michel = np.arange(40, 500, bin_size_Michel)   # default: 60, 500
xlim_low_Michel = 0; xlim_high_Michel = 650            # default plot window: [20, 650]

# # # # # # # # # # # # # # # # # # # # # # # # # #
print('\n------- Michel plot -------\n')

balanced_MC, balanced_data = balance_samples(total_charge, CPE_data)

assert len(balanced_MC) == len(balanced_data), 'Sampling failed to balance datasets!'

if verbose:
    print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


counts_data, bins_data = np.histogram(balanced_data, bins=binning_Michel)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_Michel)
counts_mc = raw_counts_mc

fill_bins = generate_merged_bins(binning_Michel, counts_data, counts_mc, min_counts=10)

counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
counts_mc = raw_counts_mc

if verbose:
    print('\nMerged bin edges:')
    for i in range(len(fill_bins) - 1):
        print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
data_errors = np.sqrt(counts_data)
mc_errors = np.sqrt(raw_counts_mc)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) /
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
ndf = len(counts_data) - 1
print(f'\nChi2/NDF: {chi2 / ndf:.2f}')


mc_data = []; mc_errors_plot = []; x_info = []

for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue
    x_info.append(binscenters[i])

    if counts_mc[i] == 0:
        mc_data.append(0)
        mc_errors_plot.append(0)
    else:
        ratio_mc = counts_mc[i] / counts_data[i]
        error_mc = ratio_mc * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors[i]   / counts_mc[i])**2
        )
        mc_data.append(ratio_mc)
        mc_errors_plot.append(error_mc)


# ---------- plotting ----------
fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data Michels', fontsize=16)
ax2.set_xlabel('Cluster charge [p.e.]', fontsize=14)
ax1.set_ylabel(f'clusters / {bin_size_Michel} p.e. bin', fontsize=14)
ax2.set_ylabel('MC / Data', fontsize=14)

ax1.set_xlim([xlim_low_Michel, xlim_high_Michel])
ax2.set_xlim([xlim_low_Michel, xlim_high_Michel]); ax2.set_ylim([0, 2])

ax1.hist(binscenters, bins=fill_bins, weights=counts_mc,
         histtype='step', color=plot_color_Michel, label='MC', linewidth=2)

# Pad arrays to match the length of fill_bins (N+1) so every (re-)merged bin gets an error band
counts_mc_padded = np.append(counts_mc, counts_mc[-1])
mc_errors_padded = np.append(mc_errors, mc_errors[-1])
mc_low  = counts_mc_padded - mc_errors_padded
mc_high = counts_mc_padded + mc_errors_padded

ax1.fill_between(fill_bins, mc_low, mc_high, step='post', color=plot_color_Michel, alpha=0.3)

ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize=1,
             fmt='o', color=plot_color_Michel)
ax2.axhline(1, linestyle='dashed', color='k')

ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}",
         transform=ax1.transAxes, fontsize=10, color=plot_color_Michel)
ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=10)
ax1.legend(fontsize=12, frameon=False, loc='upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_Michels_total_cluster_pe_' + plot_save_name_base + '_CCB_0.18_MC.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('\ndone')


------- Michel plot -------


Chi2/NDF: 1.03

done


### 3d. Throughgoing muon — cumulative cluster charge

In [22]:
# Through-going plot

# IT IS EXPECTED THIS DOES NOT ALIGN - SEE SD DISSERTATION

plot_save_dir = 'plots/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False

plot_color_thru = 'tab:red'
bin_size_thru = 100     # default 100
binning_thru = np.arange(500, 6500, bin_size_thru)   # default: 500, 6500
xlim_low_thru = 0; xlim_high_thru = 8000             # default plot window: [0, 8000]

# # # # # # # # # # # # # # # # # # # # # # # # # #
print('\n------- Throughgoing plot -------\n')

balanced_MC, balanced_data = balance_samples(total_charge_thru, CPE_thru_data)

assert len(balanced_MC) == len(balanced_data), 'Sampling failed to balance datasets!'

if verbose:
    print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


counts_data, bins_data = np.histogram(balanced_data, bins=binning_thru)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_thru)
counts_mc = raw_counts_mc

fill_bins = generate_merged_bins(binning_thru, counts_data, counts_mc, min_counts=10)

counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
counts_mc = raw_counts_mc

if verbose:
    print('\nMerged bin edges:')
    for i in range(len(fill_bins) - 1):
        print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
data_errors = np.sqrt(counts_data)
mc_errors = np.sqrt(raw_counts_mc)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) /
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
ndf = len(counts_data) - 1
print(f'\nChi2/NDF: {chi2 / ndf:.2f}')


mc_data = []; mc_errors_plot = []; x_info = []

for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue
    x_info.append(binscenters[i])

    if counts_mc[i] == 0:
        mc_data.append(0)
        mc_errors_plot.append(0)
    else:
        ratio_mc = counts_mc[i] / counts_data[i]
        error_mc = ratio_mc * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors[i]   / counts_mc[i])**2
        )
        mc_data.append(ratio_mc)
        mc_errors_plot.append(error_mc)


# ---------- plotting ----------
fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data Throughgoing Muons', fontsize=16)
ax2.set_xlabel('Cluster charge [p.e.]', fontsize=14)
ax1.set_ylabel(f'clusters / {bin_size_thru} p.e. bin', fontsize=14)
ax2.set_ylabel('MC / Data', fontsize=14)

ax1.set_xlim([xlim_low_thru, xlim_high_thru])
ax2.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_ylim([0, 2])

ax1.hist(binscenters, bins=fill_bins, weights=counts_mc,
         histtype='step', color=plot_color_thru, label='MC', linewidth=2)

counts_mc_padded = np.append(counts_mc, counts_mc[-1])
mc_errors_padded = np.append(mc_errors, mc_errors[-1])
mc_low  = counts_mc_padded - mc_errors_padded
mc_high = counts_mc_padded + mc_errors_padded

ax1.fill_between(fill_bins, mc_low, mc_high, step='post', color=plot_color_thru, alpha=0.3)

ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize=1,
             fmt='o', color=plot_color_thru)
ax2.axhline(1, linestyle='dashed', color='k')

ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}",
         transform=ax1.transAxes, fontsize=10, color=plot_color_thru)
ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=10)
ax1.legend(fontsize=12, frameon=False, loc='upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_Thru_cluster_pe_' + plot_save_name_base + '.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('\ndone')


------- Throughgoing plot -------


Chi2/NDF: 15.46

done


## 4. N-trial chi² sampling (AmBe cumulative)

Repeat the random-sampling chi² fit many times to get a median value and reduce single-trial sampling bias. Run the previous cell (3a) to set `binning_AmBe` before running this.

In [23]:
'''Same as 3a (cluster charge) but no plotting - just run the chi sq fitting N times for an average / median value.

For PMT hits, re-define binning_AmBe to the hits binning before running.'''

# rather than scaling the MC counts to data, we randomly sample. We should therefore perform an N
# trial chi-sq agreement to reduce shift / bias from MC stats

total_x2_approx = []

verbose = False

N_trials = 100

# # # # # # # # # # # # # # # # # # # # # # # # # #

for jx in trange(N_trials):

    data_dict = defaultdict(list)
    mc_dict_approx_tilt = defaultdict(list)

    data_all = []; mc_all_approx_tilt = []

    # Key: (x, y, z)
    for i in range(len(cluster_charge)):
        key = (source_position[0][i], source_position[1][i], source_position[2][i])
        data_dict[key].append(cluster_charge[i])

    for i in range(len(cluster_charge_MC)):
        key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
        mc_dict_approx_tilt[key].append(cluster_charge_MC[i])

    mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
    data_keys = list(data_dict.keys())

    valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

    min_data = min(len(data_dict[k]) for k in valid_keys)
    min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
    min_common = min(min_data, min_mc_approx_tilt)

    data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
    mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

    assert len(data_all) == len(mc_all_approx_tilt), 'Data and MC not balanced after enforced key matching!'

    if verbose:
        print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


    counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning_AmBe, counts_data, counts_mc_approx_tilt, min_counts=10)

    counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    if verbose:
        print('\nMerged bin edges:')
        for i in range(len(fill_bins) - 1):
            print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


    xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1
    if verbose:
        print(f'\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}')

    total_x2_approx.append(chi2_approx_tilt / ndf)


print('\ndone')

100%|███████████████████████████████████████| 1000/1000 [02:24<00:00,  6.91it/s]


done


In [24]:
# Plot the chi2 distribution

print(np.median(total_x2_approx))

fig, ax1 = plt.subplots(figsize=(4, 3))
plt.hist(total_x2_approx, bins=10, histtype='step', color='dodgerblue',
         label='Approx. geometry', linewidth=2)
plt.xlabel(r'$\chi^2$' + '/dof', loc='right')
plt.title('AmBe MC fit')
ax1.text(0.58, 0.7, f"$\chi^2$/ndf = {round(np.median(total_x2_approx),2)}",
         transform=ax1.transAxes, fontsize=12, color='dodgerblue')
plt.tight_layout()
#plt.savefig('plots/chi2_fit_results_cumulative_charge_1000_trials.png',
#            dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

3.3509910906044587


## 5. AmBe — by source position

Five-by-four grid (one panel per (port, depth)) and a one-by-five row (summed over ports for each depth) give a position-resolved view of agreement.

### 5a. AmBe — grid by source position

In [25]:
# 4 ports x 5 depths grid

# [0,100,102,0] (remove Port 3 +100cm)

labels = [
    "Port 5, Y = +100cm", "Port 5, Y = +50cm", "Port 5, Y = 0cm", "Port 5, Y = -50cm", "Port 5, Y = -100cm",
    "Port 1, Y = +100cm", "Port 1, Y = +50cm", "Port 1, Y = 0cm", "Port 1, Y = -50cm", "Port 1, Y = -100cm",
    "Port 4, Y = +100cm", "Port 4, Y = +50cm", "Port 4, Y = 0cm", "Port 4, Y = -50cm", "Port 4, Y = -100cm",
    "Port 3, Y = +100cm", "Port 3, Y = +50cm", "Port 3, Y = 0cm", "Port 3, Y = -50cm", "Port 3, Y = -100cm"
]

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
           [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],  # Port 1
           [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],       # Port 4
           [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]  # Port 3


geo_colors = 'dodgerblue'

fig, axes = plt.subplots(4, 5, figsize=(20, 12))

for sp in range(len(poss)):

    # default
    bin_size = 4
    binning = np.arange(5, 69, bin_size)

    # Watchboy / HM
    #bin_size = 2
    #binning = np.arange(0, 28, bin_size)

    print('\n\n############################')
    print(labels[sp])

    row = sp // 5
    col = sp % 5
    ax = axes[row, col]

    # Port 3 +100cm is blank (no data)
    if labels[sp] == "Port 3, Y = +100cm":
        ax.axis('off')
        continue

    # extract full data + MC samples for this position
    data_ = [cluster_charge[i] for i in range(len(cluster_charge)) if
             source_position[0][i] == poss[sp][0] and
             source_position[1][i] == poss[sp][1] and
             source_position[2][i] == poss[sp][2]]

    mc_approx_tilt_ = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if
                       source_position_MC[0][i] == MC_poss[sp][0] and
                       source_position_MC[1][i] == MC_poss[sp][1] and
                       source_position_MC[2][i] == MC_poss[sp][2]]

    # equalize entries per position via random sampling
    nmin = min(len(data_), len(mc_approx_tilt_))
    if nmin == 0:
        continue

    data_ = random.sample(data_, nmin)
    mc_approx_tilt_ = random.sample(mc_approx_tilt_, nmin)

    if len(data_) == 0 or len(mc_approx_tilt_) == 0:
        continue


    counts_data, bins_data = np.histogram(data_, bins=binning)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

    counts_data, bins_data = np.histogram(data_, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]

    # MC errors: sqrt(raw counts), then scaled
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)

    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])

    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1
    print(f'\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}')


    # ---------- plot ----------
    ax.set_title(labels[sp], fontsize=12)
    ax.set_xlabel('Cluster charge [p.e.]')
    ax.set_ylabel('clusters / ' + str(bin_size) + ' p.e. bin', fontsize=12)
    ax.set_xlim([0, 80])
    #ax.set_xlim([0, 35])   # WB/HM

    ax.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt,
            histtype='step', color=geo_colors, linewidth=2)
    ax.fill_between(
        binscenters,
        counts_mc_approx_tilt - mc_errors_approx_tilt,
        counts_mc_approx_tilt + mc_errors_approx_tilt,
        step='mid', color=geo_colors, alpha=0.3
    )

    ax.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')
    ax.text(0.6, 0.85, 'entries = ' + str(sum(counts_data)), fontsize=9, transform=ax.transAxes)
    ax.text(0.6, 0.75, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}",
            transform=ax.transAxes, fontsize=9, color=geo_colors)


# Hide empty subplots beyond the last one we touched
for i in range(sp + 1, len(axes)):
    fig.delaxes(axes[i])

legend_elements = [
    Line2D([0], [0], color='k',         marker='o', linestyle='None', label='Data'),
    Line2D([0], [0], color=geo_colors,              linestyle='-',    label='MC'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=24,
           frameon=False, bbox_to_anchor=(0.5, 1.05))

plt.tight_layout()
path_str = 'plots/AmBe_total_cluster_pe_all_source_positions_QE_1.50_HM_1.25.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()



############################
Port 5, Y = +100cm

Approx tilt Chi2/NDF: 1.13


############################
Port 5, Y = +50cm

Approx tilt Chi2/NDF: 4.79


############################
Port 5, Y = 0cm

Approx tilt Chi2/NDF: 1.09


############################
Port 5, Y = -50cm

Approx tilt Chi2/NDF: 2.26


############################
Port 5, Y = -100cm

Approx tilt Chi2/NDF: 1.31


############################
Port 1, Y = +100cm

Approx tilt Chi2/NDF: 1.79


############################
Port 1, Y = +50cm

Approx tilt Chi2/NDF: 2.41


############################
Port 1, Y = 0cm

Approx tilt Chi2/NDF: 2.26


############################
Port 1, Y = -50cm

Approx tilt Chi2/NDF: 2.09


############################
Port 1, Y = -100cm

Approx tilt Chi2/NDF: 5.84


############################
Port 4, Y = +100cm

Approx tilt Chi2/NDF: 1.70


############################
Port 4, Y = +50cm

Approx tilt Chi2/NDF: 1.90


############################
Port 4, Y = 0cm

Approx tilt Chi2/NDF: 1.95


### 5b. AmBe — summed across ports at each depth

In [26]:
# 5 depth columns, summed across ports

depths_cm = [+100, +50, 0, -50, -100]
depth_labels = [f"Y = {y} cm" for y in depths_cm]
depth_groups = {y: [] for y in depths_cm}
for i, p in enumerate(poss):
    depth_groups[p[1]].append(i)

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],
           [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],
           [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],
           [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]


fig = plt.figure(figsize=(20, 4))
outer_gs = gridspec.GridSpec(1, 5, wspace=0.5)

for d, y in enumerate(depths_cm):

    # default
    bin_size = 2
    binning = np.arange(5, 71, bin_size)

    # Watchboy / HM
    #bin_size = 1
    #binning = np.arange(0, 31, bin_size)

    print('\n############################')
    print(depth_labels[d], '\n')

    data_entries_all = []
    mc_entries_approx_all = []

    # samples per position for this depth
    for sp in depth_groups[y]:

        if MC_poss[sp][0] == 0 and MC_poss[sp][1] == 100 and MC_poss[sp][2] == 102:
            print('\nPort 3 +100cm NOT INCLUDED')
            continue

        data_entries = [cluster_charge[i] for i in range(len(cluster_charge)) if
                        source_position[0][i] == poss[sp][0] and
                        source_position[1][i] == poss[sp][1] and
                        source_position[2][i] == poss[sp][2]]

        mc_approx_tilt_entries = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if
                                  source_position_MC[0][i] == MC_poss[sp][0] and
                                  source_position_MC[1][i] == MC_poss[sp][1] and
                                  source_position_MC[2][i] == MC_poss[sp][2]]

        if len(data_entries) > 0 or len(mc_approx_tilt_entries):
            data_entries_all.append(data_entries)
            mc_entries_approx_all.append(mc_approx_tilt_entries)


    # Random subsample per-position to ensure balanced contribution
    data_ = []
    mc_approx_tilt_ = []
    for data_entries, mc_entries_approx in zip(data_entries_all, mc_entries_approx_all):
        nmin = min(len(data_entries), len(mc_entries_approx))
        if nmin == 0:
            continue
        data_           += random.sample(data_entries,         nmin)
        mc_approx_tilt_ += random.sample(mc_entries_approx,    nmin)

    assert len(data_) == len(mc_approx_tilt_), f'Mismatch at depth {y} cm'


    counts_data, bins_data = np.histogram(data_, bins=binning)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

    counts_data, bins_data = np.histogram(data_, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)
    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)

    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])

    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1
    print(f'\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}')


    # residuals
    mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; x_info = []
    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue
        x_info.append(binscenters[i])

        if counts_mc_approx_tilt[i] == 0:
            mc_approx_tilt_data.append(0)
            mc_errors_approx_tilt_plot.append(0)
        else:
            ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
            error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
            )
            mc_approx_tilt_data.append(ratio_mc_approx_tilt)
            mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)


    inner_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[d],
                                                height_ratios=[3, 1], hspace=0.35)
    ax1 = fig.add_subplot(inner_gs[0])
    ax2 = fig.add_subplot(inner_gs[1])

    ax1.set_title(depth_labels[d], fontsize=12)
    ax1.set_ylabel('clusters / ' + str(bin_size) + ' p.e. bin', fontsize=12)
    ax1.set_xlim([0, 80])
    #ax1.set_xlim([0, 35])  # WB / HM

    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt,
             histtype='step', color=geo_colors, linewidth=2)
    ax1.fill_between(
        binscenters,
        counts_mc_approx_tilt - mc_errors_approx_tilt,
        counts_mc_approx_tilt + mc_errors_approx_tilt,
        step='mid', color=geo_colors, alpha=0.3
    )

    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')
    ax1.text(0.54, 0.9, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=8)
    ax1.text(0.54, 0.8, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}",
             transform=ax1.transAxes, fontsize=8, color=geo_colors)

    ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot,
                 fmt='o', color=geo_colors, markersize=3)
    ax2.axhline(1, linestyle='dashed', color='k')
    ax2.set_ylim([0, 2])
    ax2.set_xlim([0, 80])
    #ax2.set_xlim([0, 35])  # WB / HM
    ax2.set_ylabel('MC / Data', fontsize=10)
    ax2.set_xlabel('cluster charge [p.e.]', fontsize=12)

    ax1.xaxis.set_major_locator(MultipleLocator(20))
    ax2.xaxis.set_major_locator(MultipleLocator(20))


legend_elements = [
    Line2D([0], [0], color='k',         marker='o', linestyle='None', label='Data'),
    Line2D([0], [0], color=geo_colors,              linestyle='-',    label='MC'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=24,
           frameon=False, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
path_str = 'plots/AmBe_total_cluster_pe_QE_1.50_HM_1.25.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()


############################
Y = 100 cm 


Port 3 +100cm NOT INCLUDED

Approx tilt Chi2/NDF: 1.77

############################
Y = 50 cm 


Approx tilt Chi2/NDF: 3.97

############################
Y = 0 cm 


Approx tilt Chi2/NDF: 2.62

############################
Y = -50 cm 


Approx tilt Chi2/NDF: 1.77

############################
Y = -100 cm 


Approx tilt Chi2/NDF: 1.98


/var/folders/nf/wq25_wtn08vb7t3wg0hydmhr0000gr/T/ipykernel_9012/268332760.py:155: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


### 5c. AmBe — N-trial chi² sampling per (port, depth)

Populates `x2_approx[sp]` (sp = 0..19) with 100 chi² values per position for the per-port plot below.

In [29]:
# chisq sampling per position (source and depth)

# [0,100,102,0] (remove Port 3 +100cm)

labels = [
    "Port 5, Y = +100cm", "Port 5, Y = +50cm", "Port 5, Y = 0cm", "Port 5, Y = -50cm", "Port 5, Y = -100cm",
    "Port 1, Y = +100cm", "Port 1, Y = +50cm", "Port 1, Y = 0cm", "Port 1, Y = -50cm", "Port 1, Y = -100cm",
    "Port 4, Y = +100cm", "Port 4, Y = +50cm", "Port 4, Y = 0cm", "Port 4, Y = -50cm", "Port 4, Y = -100cm",
    "Port 3, Y = +100cm", "Port 3, Y = +50cm", "Port 3, Y = 0cm", "Port 3, Y = -50cm", "Port 3, Y = -100cm"
]

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],
           [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],
           [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],
           [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]


x2_approx = [[], [], [], [], [],   # Port 5
             [], [], [], [], [],   # Port 1
             [], [], [], [], [],   # Port 4
             [], [], [], [], []]   # Port 3 (+100 will be empty)

N_trials = 100

for jk in trange(N_trials, desc = 'Running trials...'):
    for sp in range(len(poss)):

        # default
        bin_size = 4
        binning = np.arange(5, 69, bin_size)

        # Watchboy / HM
        #bin_size = 2
        #binning = np.arange(0, 28, bin_size)

        # skip Port 3 +100 cm
        if labels[sp] == "Port 3, Y = +100cm":
            continue

        # extract full data + MC samples for this position
        data_ = [cluster_charge[i] for i in range(len(cluster_charge)) if
                 source_position[0][i] == poss[sp][0] and
                 source_position[1][i] == poss[sp][1] and
                 source_position[2][i] == poss[sp][2]]

        mc_approx_tilt_ = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if
                           source_position_MC[0][i] == MC_poss[sp][0] and
                           source_position_MC[1][i] == MC_poss[sp][1] and
                           source_position_MC[2][i] == MC_poss[sp][2]]

        nmin = min(len(data_), len(mc_approx_tilt_))
        if nmin == 0:
            continue

        data_ = random.sample(data_, nmin)
        mc_approx_tilt_ = random.sample(mc_approx_tilt_, nmin)

        if len(data_) == 0 or len(mc_approx_tilt_) == 0:
            continue


        counts_data, bins_data = np.histogram(data_, bins=binning)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)
        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

        counts_data, bins_data = np.histogram(data_, bins=fill_bins)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)
        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
        data_errors = np.sqrt(counts_data)
        mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)

        binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])

        chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) /
                                  (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
        ndf = len(counts_data) - 1
        x2_approx[sp].append(chi2_approx_tilt / ndf)


print('\ndone')

Running trials...: 100%|██████████████████████| 100/100 [01:53<00:00,  1.14s/it]


done


### 5d. Plot chi² vs depth, broken out by port

In [30]:
# chi_sq plot (per source position, grouped by port)

depths_cm = [+100, +50, 0, -50, -100]
chisq_dof_approx_tilt = []
chisq_dof_approx_tilt_er = []
for i in range(len(x2_approx)):
    chisq_dof_approx_tilt.append(np.median(x2_approx[i]))  # median of 100 samples
    chisq_dof_approx_tilt_er.append(np.std(x2_approx[i]))

# overall median across all positions (manual reference value -- update for your run)
median_approx_tilt = 1.74

# Watchboys
#median_no_tilt = 32.37
#median_approx_tilt = 5.15

plt.figure(figsize=(4, 3))

plt.axhline(median_approx_tilt, linestyle='dashed', color='black', label='Total fit')

print('PORT 5:', labels[0:5],  '\n')
print('PORT 1:', labels[5:10], '\n')
print('PORT 4:', labels[10:15],'\n')
print('PORT 3:', labels[16:20],'\n')

plt.errorbar(depths_cm,    chisq_dof_approx_tilt[0:5],   yerr=chisq_dof_approx_tilt_er[0:5],
             fmt='o-', color='tab:orange', capsize=3, label='Port 5')
plt.errorbar(depths_cm,    chisq_dof_approx_tilt[5:10],  yerr=chisq_dof_approx_tilt_er[5:10],
             fmt='o-', color='tab:red',    capsize=3, label='Port 1')
plt.errorbar(depths_cm,    chisq_dof_approx_tilt[10:15], yerr=chisq_dof_approx_tilt_er[10:15],
             fmt='o-', color='tab:blue',   capsize=3, label='Port 4')
plt.errorbar(depths_cm[1:5], chisq_dof_approx_tilt[16:20], yerr=chisq_dof_approx_tilt_er[16:20],
             fmt='o-', color='tab:green',  capsize=3, label='Port 3')

plt.xlabel('Source Depth [cm]')
plt.ylabel(r'$\chi^{2} / dof$')
plt.title('AmBe neutrons')
plt.legend(fontsize=12, frameon=False, ncol=2)
plt.ylim([0, 10])
plt.gca().invert_xaxis()
plt.tight_layout()
path_str = 'plots/100_TRIALS_ALL_POS_CHISQ_AmBe_total_cluster_pe_QE_1.50_HM_1.25.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('done')

PORT 5: ['Port 5, Y = +100cm', 'Port 5, Y = +50cm', 'Port 5, Y = 0cm', 'Port 5, Y = -50cm', 'Port 5, Y = -100cm'] 

PORT 1: ['Port 1, Y = +100cm', 'Port 1, Y = +50cm', 'Port 1, Y = 0cm', 'Port 1, Y = -50cm', 'Port 1, Y = -100cm'] 

PORT 4: ['Port 4, Y = +100cm', 'Port 4, Y = +50cm', 'Port 4, Y = 0cm', 'Port 4, Y = -50cm', 'Port 4, Y = -100cm'] 

PORT 3: ['Port 3, Y = +50cm', 'Port 3, Y = 0cm', 'Port 3, Y = -50cm', 'Port 3, Y = -100cm'] 

done


/Users/doran/opt/anaconda3/lib/python3.9/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/doran/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/doran/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/doran/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/doran/opt/anaconda3/lib/python3.9/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## 6. chi² scan summary plots

The plots below summarize chi² scans over the photodetection efficiency factor `r_CE` (and PMT type). The numerical values are hardcoded from previous runs of the comparison cells in section 3 -- update them afterrunning a new scan.

### 6a. Overall chi²/dof across samples

In [31]:
# chi_sq plot -- AmBe + Michel + Thru on one axis

QE_vals = [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15]
chisq_dof_AmBe   = [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57]
chisq_dof_Michel = [124/11, 46/12, 22/12, 42/12, 85/12, 155/12, 208/12]

QE_vals_thru   = [0.80, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1]
chisq_dof_thru = [11.05, 3.36, 2.69, 6.42, 16.06, 25.73, 36.07]

plt.figure(figsize=(4, 3))
plt.plot(QE_vals[1:],  chisq_dof_AmBe[1:], marker='o', linestyle='-', color='tab:purple', label='AmBe')
plt.plot(QE_vals,      chisq_dof_Michel,   marker='o', linestyle='-', color='tab:orange', label='Michels')
plt.plot(QE_vals_thru, chisq_dof_thru,     marker='o', linestyle='-', color='tab:red',    label='Thru-' + r'$\mu$')

plt.xlabel(r'$r_{CE}$')
plt.ylabel(r'$\chi^{2} / dof$')
plt.legend(fontsize=12, frameon=False)
plt.tight_layout()
path_str = 'plots/CHISQ _ AMBE and MICHEL AND THRU _ QE_1.50_HM_1.25.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

### 6b. chi²/dof per PMT type 

(HM shown; LUX + ETEL + WB commented)

In [34]:
# chi_sq plot per PMT type

def plot_chi2(QE_vals, chisqdof, labels, colors, TYPE):

    plt.figure(figsize=(4, 3))

    for i in range(len(QE_vals)):
        plt.plot(QE_vals[i], chisqdof[i], marker='o', linestyle='-', color=colors[i], label=labels[i])

    plt.xlabel(r'$r_{CE}$')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize=12, frameon=True, loc='upper right')
    plt.title(TYPE, fontsize=12)
    plt.tight_layout()
    path_str = 'plots/CHISQ ' + TYPE + ' _ AMBE and MICHEL AND THRU _ QE_1.50_HM_1.25.png'
    #plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.show()


labels = ['AmBe', 'Michel', 'Thru-' + r'$\mu$']
colors = ['tab:purple', 'tab:orange', 'tab:red']


# --- Watchboy ---
QE_vals_WB = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],    # AmBe
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],    # Michels
    [0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]  # Thru
]
chisq_WB = [
    [26.57, 13.58, 9.08, 6.16, 7.65, 12.6, 19.35],            # AmBe
    [7.06, 4.06, 2.35, 2.22, 2.1, 4.04, 5.49],                # Michels
    [37.8, 16.83, 6.45, 6.68, 15.48, 26.05, 41.04, 54.86]     # Thru
]
#plot_chi2(QE_vals_WB, chisq_WB, labels, colors, 'WATCHBOY')


# --- Hamamatsu ---
QE_vals_HM = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],   # AmBe
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],   # Michels
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15]         # Thru
]
chisq_HM = [
    [27.83, 15.76, 9.01, 3.56, 1.51, 2.69, 7.18, 11.48],  # AmBe
    [8.79,  3.85,  2.33, 0.96, 1.12, 2.14, 2.76, 4.99],   # Michels
    [25.17, 12.09, 4.8,  2.53, 6.85, 14.58, 22.22]        # Thru
]
#plot_chi2(QE_vals_HM, chisq_HM, labels, colors, 'HAMAMATSU')


# --- LUX + ETEL ---
QE_vals_LE = [
    [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],     # AmBe
    [0.75, 0.80, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1],         # Michels
    [0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]                # Thru
]
chisq_LE = [
    [8.06, 5.01, 3.93, 1.48, 1.63, 2.61, 4.13, 5.81, 8.21],  # AmBe
    [4.83, 2.29, 1.52, 1.33, 2.20, 2.41, 4.45, 6.41],        # Michels
    [6.79, 3.66, 1.87, 2.31, 2.71, 4.44, 7.45]               # Thru
]

# adjust types as needed
plot_chi2(QE_vals_HM, chisq_HM, labels, colors, 'Hamamatsu')
#plot_chi2(QE_vals_LE, chisq_LE, labels, colors, 'LUX + ETEL')


print('\nPlots saved')


Plots saved


### 6c. AmBe chi²/dof — total + per PMT type

In [35]:
# AmBe chisq fit -- per PMT type

def plot_chi2_AmBe(QE_vals, chisqdof, labels, linestyles, linewidths, markers, colors, TYPE):

    plt.figure(figsize=(5, 3))

    for i in range(len(QE_vals)):
        plt.plot(QE_vals[i], chisqdof[i], marker='o', linestyle=linestyles[i],
                 color=colors[i], label=labels[i], linewidth=linewidths[i])

    plt.xlabel(r'$r_{CE}$')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.title(TYPE, fontsize=14)
    plt.tight_layout()
    path_str = 'plots/CHISQ ' + TYPE + ' _ AMBE _ QE_1.50_HM_1.25.png'
    #plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.show()


labels     = ['Total', 'HM', 'WB', 'LUX + ETEL']
linestyles = ['-', 'dashed', 'dashed', 'dashed']
linewidths = [2, 1, 1, 1]
markers    = ['o', 'd', 's', 'v']
colors     = ['black', 'tab:purple', 'dodgerblue', 'tab:red']


chisq_dof_AmBe = [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57]


QE_vals = [
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],                # total
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2],           # HM
    [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15],                # WB
    [0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2]       # ETEL + LUX
]

chisq = [
    [2050/55, 991/58, 312/58, 82/57, 251/57, 680/57, 1274/57],          # total
    [27.83, 15.76, 9.01, 3.56, 1.51, 2.69, 7.18, 11.48],                # HM
    [26.57, 13.58, 9.08, 6.16, 7.65, 12.6, 19.35],                      # WB
    [8.06, 5.01, 3.93, 1.48, 1.63, 2.61, 4.13, 5.81, 8.21]              # ETEL + LUX
]

plot_chi2_AmBe(QE_vals, chisq, labels, linestyles, linewidths, markers, colors, 'AmBe neutrons')


print('\nPlots saved')


Plots saved


### 6d. AmBe chi²/dof — by port position and depth

Default geometry shown

In [37]:
# AmBe chisq -- by Port position and depth

def plot_chi2_AmBe(chisqdof, labels, colors, averages):

    plt.figure(figsize=(5, 3))

    depths_cm = [+100, +50, 0, -50, -100]
    for i in range(len(chisqdof)):
        if i == 3:   # Port 3 omitted Y = +100
            plt.plot([+50, 0, -50, -100], chisqdof[i], marker='o', linestyle='-',
                     color=colors[i], label=labels[i], linewidth=1)
        else:
            plt.plot(depths_cm, chisqdof[i], marker='o', linestyle='-',
                     color=colors[i], label=labels[i], linewidth=1)

    plt.plot(depths_cm, averages, color='k', zorder=20, marker='+',
             label='Cumulative', linestyle='dashed')

    plt.xlabel('Source Depth [cm]')
    plt.ylabel(r'$\chi^{2} / dof$')
    plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.ylim([0, 25])
    plt.title('AmBe neutrons (Default Tilt)', fontsize=14)
    plt.gca().invert_xaxis()
    plt.tight_layout()
    path_str = 'plots/CHISQ DEFAULT TILTING _ AMBE _ BY POSITION _ QE_1.50_HM_1.25.png'
    #plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
    plt.show()


labels = ['Port 5', 'Port 1', 'Port 4', 'Port 3']
colors = ['tab:orange', 'dodgerblue', 'tab:purple', 'tab:red']

'''# approx tilted
chisq_dof_AmBe = [[12/14, 76/14, 16/13, 33/14, 11/13],   # Port 5
                  [53/14, 51/14, 33/14, 46/14, 74/13],   # Port 1
                  [20/14, 16/14, 22/14, 24/14, 31/13],   # Port 4
                         [30/14, 32/14, 44/13, 24/12]    # Port 3 (+100 omitted)
                 ]
averages = [44/30, 121/31, 68/31, 69/30, 46/29]
#'''

#''' # default
chisq_dof_AmBe = [[26/14, 267/14, 95/13, 280/14, 213/13],  # Port 5
                  [25/14,  66/14, 310/14, 22/14,  48/13],  # Port 1
                  [15/14,  39/14, 119/14, 34/14, 207/13],  # Port 4
                         [32/14,  27/14, 86/13,  38/12]    # Port 3 (+100 omitted)
                 ]
averages = [45/30, 310/31, 533/31, 296/30, 371/29]
#'''


plot_chi2_AmBe(chisq_dof_AmBe, labels, colors, averages)


print('done')

done


## 7. Thru-muon investigations (optional)

Additional thru-muon checks: upstream/downstream charge ratio comparisonbetween MC and data, useful for diagnosing tilt and angular response.

In [38]:
# upstream / downstream charge ratio comparison

plot_save_dir = 'plots/'
plot_save_name_base = 'QE_1.50_HM_1.25'

verbose = False

plot_color_thru = 'tab:red'
bin_size_thru = 0.05
binning_thru = np.arange(0.1, 0.95, bin_size_thru)
xlim_low_thru = 0; xlim_high_thru = 1.00

# # # # # # # # # # # # # # # # # # # # # # # # # #

# create the ratio
dummy = []; dummy_data = []
for i in range(len(upstream_charge)):
    dummy.append(upstream_charge[i] / downstream_charge[i])
for i in range(len(Up_thru_data)):
    dummy_data.append(Up_thru_data[i] / Down_thru_data[i])


print('\n------- Throughgoing plot -------\n')

balanced_MC, balanced_data = balance_samples(dummy, dummy_data)

assert len(balanced_MC) == len(balanced_data), 'Sampling failed to balance datasets!'

if verbose:
    print('\nCounts after sampling:', len(balanced_MC), len(balanced_data), '\n')


counts_data, bins_data = np.histogram(balanced_data, bins=binning_thru)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=binning_thru)
counts_mc = raw_counts_mc

fill_bins = generate_merged_bins(binning_thru, counts_data, counts_mc, min_counts=10)

counts_data, bins_data = np.histogram(balanced_data, bins=fill_bins)
raw_counts_mc, bins_mc = np.histogram(balanced_MC, bins=fill_bins)
counts_mc = raw_counts_mc

if verbose:
    print('\nMerged bin edges:')
    for i in range(len(fill_bins) - 1):
        print(f'Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})')


xwidth = [(fill_bins[i+1] - fill_bins[i]) / 2 for i in range(len(fill_bins) - 1)]
data_errors = np.sqrt(counts_data)
mc_errors = np.sqrt(raw_counts_mc)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins) - 1)])


chi2 = np.sum(((np.array(counts_data) - np.array(counts_mc)) ** 2) /
              (np.array(data_errors) ** 2 + np.array(mc_errors) ** 2))
ndf = len(counts_data) - 1
print(f'\nChi2/NDF: {chi2 / ndf:.2f}')


mc_data = []; mc_errors_plot = []; x_info = []

for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue
    x_info.append(binscenters[i])

    if counts_mc[i] == 0:
        mc_data.append(0)
        mc_errors_plot.append(0)
    else:
        ratio_mc = counts_mc[i] / counts_data[i]
        error_mc = ratio_mc * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors[i]   / counts_mc[i])**2
        )
        mc_data.append(ratio_mc)
        mc_errors_plot.append(error_mc)


# ---------- plotting ----------
fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data Throughgoing Muons', fontsize=16)
ax2.set_xlabel('upstream / downstream charge ratio', fontsize=14)
ax1.set_ylabel(f'clusters / {bin_size_thru} p.e. bin', fontsize=14)
ax2.set_ylabel('MC / Data', fontsize=14)

ax1.set_xlim([xlim_low_thru, xlim_high_thru])
ax2.set_xlim([xlim_low_thru, xlim_high_thru]); ax2.set_ylim([0, 2])

ax1.hist(binscenters, bins=fill_bins, weights=counts_mc,
         histtype='step', color=plot_color_thru, label='MC', linewidth=2)

counts_mc_padded = np.append(counts_mc, counts_mc[-1])
mc_errors_padded = np.append(mc_errors, mc_errors[-1])
mc_low  = counts_mc_padded - mc_errors_padded
mc_high = counts_mc_padded + mc_errors_padded

ax1.fill_between(fill_bins, mc_low, mc_high, step='post', color=plot_color_thru, alpha=0.3)

ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

ax2.errorbar(x_info, mc_data, xerr=xwidth, yerr=mc_errors_plot, markersize=1,
             fmt='o', color=plot_color_thru)
ax2.axhline(1, linestyle='dashed', color='k')

ax1.text(0.7, 0.55, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}",
         transform=ax1.transAxes, fontsize=10, color=plot_color_thru)
ax1.text(0.7, 0.45, f"events = {sum(counts_data)}", transform=ax1.transAxes, fontsize=10)
ax1.legend(fontsize=12, frameon=False, loc='upper right')

plt.tight_layout()
path_str = plot_save_dir + 'upstream_MC_Data_Thru_upstream_downstream_ratio_' + plot_save_name_base + '.png'
#plt.savefig(path_str, dpi=300, bbox_inches='tight', pad_inches=.3, facecolor='w')
plt.show()

print('\ndone')


------- Throughgoing plot -------


Chi2/NDF: 19.19

done


In [39]:
# Thanks for running it! :) Carry the torch forward!